In [28]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("algozee/teenager-menthal-healy")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'teenager-menthal-healy' dataset.
Path to dataset files: /kaggle/input/teenager-menthal-healy


In [29]:
import pandas as pd
df = pd.read_csv(f"{path}/Teen_Mental_Health_Dataset.csv")

In [30]:
df

,age,gender,daily_social_media_hours,platform_usage,sleep_hours,screen_time_before_sleep,academic_performance,physical_activity,social_interaction_level,stress_level,anxiety_level,addiction_level,depression_label
0,14,male,7.9,Instagram,7.4,2.9,3.01,1.5,low,2,2,1,0
1,19,female,1.9,TikTok,8.0,2.9,3.22,0.8,high,8,1,10,0
2,17,female,1.3,Instagram,7.6,0.5,3.92,0.0,high,2,4,2,0
3,15,male,7.4,TikTok,6.9,1.6,3.48,0.8,medium,1,7,9,0
4,15,female,4.7,Both,4.9,3.0,2.37,1.4,medium,3,5,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,18,female,6.8,Instagram,6.6,2.0,2.76,1.0,low,3,4,4,0
1196,16,male,2.3,Both,8.0,1.9,2.12,0.4,high,7,4,4,0
1197,14,female,1.7,Both,8.7,0.7,3.98,0.8,high,1,1,1,0
1198,15,male,3.9,Both,8.5,2.1,3.19,0.6,high,7,9,9,0


In [31]:
from sklearn.preprocessing import LabelEncoder
label = LabelEncoder()
for col in df.columns:
  df[col] = label.fit_transform(df[col])

In [32]:
df

,age,gender,daily_social_media_hours,platform_usage,sleep_hours,screen_time_before_sleep,academic_performance,physical_activity,social_interaction_level,stress_level,anxiety_level,addiction_level,depression_label
0,1,1,69,1,34,24,101,15,1,1,1,0,0
1,6,0,9,2,40,24,122,8,0,7,0,9,0
2,4,0,3,1,36,0,192,0,0,1,3,1,0
3,2,1,64,2,29,11,148,8,2,0,6,8,0
4,2,0,37,0,9,25,37,14,2,2,4,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,5,0,58,1,26,15,76,10,1,2,3,3,0
1196,3,1,13,0,40,14,12,4,0,6,3,3,0
1197,1,0,7,0,47,2,198,8,0,0,0,0,0
1198,2,1,29,0,45,16,119,6,0,6,8,8,0


In [33]:
X = df.drop(['depression_label'] , axis = 1)
Y = df['depression_label']

In [34]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [35]:
from sklearn.model_selection import train_test_split
X_train , X_val , y_train , y_val = train_test_split(X , Y , train_size = 0.8 , random_state= True)

In [44]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_val_tensor   = torch.tensor(X_val, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
y_val_tensor   = torch.tensor(y_val.values, dtype=torch.long)

In [53]:
import torch
import torch.nn as nn

class Model(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(Model, self).__init__()

        self.linear = nn.Sequential(
            nn.Linear(input_dim, output_dim)

        )

    def forward(self, x):
        return self.linear(x)

In [54]:
df.shape

(1200, 13)

In [55]:
import torch.optim as optim
model = Model(12 , 10)
optimizer = optim.Adam(model.parameters() , lr = 0.001)
criterion = nn.CrossEntropyLoss()

In [56]:
train_losses = []

patience = 5
best_loss = float('inf')
counter = 0

for epoch in range(50):
    model.train()

    optimizer.zero_grad()
    outputs = model(X_train_tensor)
    train_loss = criterion(outputs, y_train_tensor)
    train_loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val_tensor)
        val_loss = criterion(val_outputs, y_val_tensor)

    print(f"Epoch {epoch} | Train Loss: {train_loss.item()} | Val Loss: {val_loss.item()}")

    train_losses.append(train_loss.item())

    if val_loss < best_loss:
        best_loss = val_loss
        counter = 0
    else:
        counter += 1

    if counter >= patience:
        print("Early stopping triggered")
        break

Epoch 0 | Train Loss: 2.4493939876556396 | Val Loss: 2.4554567337036133
Epoch 1 | Train Loss: 2.445908784866333 | Val Loss: 2.452392816543579
Epoch 2 | Train Loss: 2.442434787750244 | Val Loss: 2.4493398666381836
Epoch 3 | Train Loss: 2.4389731884002686 | Val Loss: 2.4462976455688477
Epoch 4 | Train Loss: 2.4355227947235107 | Val Loss: 2.4432666301727295
Epoch 5 | Train Loss: 2.432084560394287 | Val Loss: 2.440246343612671
Epoch 6 | Train Loss: 2.4286587238311768 | Val Loss: 2.4372377395629883
Epoch 7 | Train Loss: 2.4252450466156006 | Val Loss: 2.4342398643493652
Epoch 8 | Train Loss: 2.421844005584717 | Val Loss: 2.4312539100646973
Epoch 9 | Train Loss: 2.4184553623199463 | Val Loss: 2.428278923034668
Epoch 10 | Train Loss: 2.41507887840271 | Val Loss: 2.4253146648406982
Epoch 11 | Train Loss: 2.411715507507324 | Val Loss: 2.4223620891571045
Epoch 12 | Train Loss: 2.4083642959594727 | Val Loss: 2.4194204807281494
Epoch 13 | Train Loss: 2.4050259590148926 | Val Loss: 2.416490077972412